# Topology-aware token-dropping end to end

This notebook is a runnable wrapper around `topology_aware_token_dropping.py`, so the CLI and notebook exercise the same implementation. Start LMCache and vLLM as described in the token-dropping README before running all cells. The final cell performs a real prefill, retrieves and edits live KV through the SDK, stores it, resumes decode, and asserts that replay against a stale request generation fails closed.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
# Standard
from pathlib import Path
from urllib.request import urlopen
import json
import os
import shlex
import subprocess
import sys

served_model = os.environ.get("VLLM_SERVED_MODEL_NAME", "Qwen/Qwen3-0.6B")
hf_model = os.environ.get("HF_MODEL_NAME", served_model)
cache_model = os.environ.get("LMCACHE_MODEL_NAME", served_model)
vllm_url = os.environ.get("VLLM_URL", "http://localhost:8000")
lmcache_url = os.environ.get("LMCACHE_URL", "http://localhost:8080")

In [ ]:
for health_url in (f"{lmcache_url}/healthcheck", f"{vllm_url}/v1/models"):
    with urlopen(health_url, timeout=10) as response:  # noqa: S310
        assert response.status == 200, (health_url, response.status)
print("LMCache and vLLM health checks passed")

In [ ]:
candidates = (
    Path("topology_aware_token_dropping.py"),
    Path("examples/token_dropping/topology_aware_token_dropping.py"),
)
script = next((path for path in candidates if path.is_file()), None)
if script is None:
    raise FileNotFoundError("run from the repository root or examples/token_dropping")
command = [
    sys.executable,
    str(script),
    "--model",
    served_model,
    "--hf-model",
    hf_model,
    "--cache-model",
    cache_model,
    "--vllm-url",
    vllm_url,
    "--lmcache-url",
    lmcache_url,
]
print("$", shlex.join(command))
completed = subprocess.run(command, check=True, capture_output=True, text=True)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
print(completed.stdout)
start = completed.stdout.find("{")
if start < 0:
    raise RuntimeError("example did not emit JSON evidence")
evidence = json.JSONDecoder().raw_decode(completed.stdout[start:])[0]
assert evidence["source_cached_tokens"] == 1024
assert evidence["kept_cached_tokens"] == 512
assert evidence["stale_generation_operations"] == 0
assert "request_generation_mismatch" in evidence["stale_generation_validation"]
assert evidence["decode_output_tokens"] > 0
print("Topology-aware live KV edit and stale-generation checks passed")